In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle

import structlog
import logging
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
)

import sys
sys.path.append('../../../../')

import pickle

from src.difsched.agents.dr3rlpy import train_bc, evaluate, train_iql, train_td3bc, train_cql
from src.difsched.agents.gym_env import HybridEnv
from src.difsched.config import getExpConfig, visualizeExpConfig
from src.difsched.env.Hybrid import createEnv

In [ ]:
trafficDatasetFolder = f'../../../../data/processed/traffic'
dataset_path = "offline_dataset.pkl"
with open(dataset_path, "rb") as f:
    dataset = pickle.load(f)

expConfigIdx = 0  # Changed from 0 to 1 to match offline dataset (N_user=20)
expParams = getExpConfig(expConfigIdx)
visualizeExpConfig(expParams)

simEnv = createEnv(expParams, trafficDatasetFolder)
simEnv.selectMode(mode="test", type="data")

In [ ]:
n_steps = 5000
n_steps_per_epoch = 100


bc = train_bc(dataset, device="cuda", n_steps=n_steps, n_steps_per_epoch=n_steps_per_epoch)
iql = train_iql(dataset, device="cuda", n_steps=n_steps, n_steps_per_epoch=n_steps_per_epoch)
cql = train_cql(dataset, device="cuda", n_steps=n_steps, n_steps_per_epoch=n_steps_per_epoch)

#td3bc = train_td3bc(dataset, device="cuda", n_steps=5000, n_steps_per_epoch=100)

# Save models to data/results/d3rlpy
save_dir = "../../../../data/results/d3rlpy"
os.makedirs(save_dir, exist_ok=True)

print(f"Saving models to {save_dir}...")
bc.save(os.path.join(save_dir, f"bc_model_exp{expConfigIdx}.d3"))
print(f"BC model saved to {os.path.join(save_dir, f'bc_model_exp{expConfigIdx}.d3')}")

iql.save(os.path.join(save_dir, f"iql_model_exp{expConfigIdx}.d3"))
print(f"IQL model saved to {os.path.join(save_dir, f'iql_model_exp{expConfigIdx}.d3')}")

cql.save(os.path.join(save_dir, f"cql_model_exp{expConfigIdx}.d3"))
print(f"CQL model saved to {os.path.join(save_dir, f'cql_model_exp{expConfigIdx}.d3')}")

print("All models saved successfully!")

In [ ]:
def evaluate_with_stats(algo, env, n_episodes=5):
    """Evaluate algorithm and return mean and std of average rewards per episode."""
    from src.difsched.agents.dr3rlpy.evaluation import reset_compat, step_compat
    import gymnasium as gym
    
    returns = []
    for _ in range(n_episodes):
        obs = reset_compat(env)
        done = False
        ep_ret = 0.0
        step_count = 0
        while not done:
            act = algo.predict(obs[None, ...])[0]
            if isinstance(env.action_space, gym.spaces.Discrete):
                act = int(act)
            obs, r, done, *_ = step_compat(env, act)
            ep_ret += r
            step_count += 1
        avg_reward = ep_ret / step_count if step_count > 0 else 0.0
        returns.append(avg_reward)
    
    mean_reward = np.mean(returns)
    std_reward = np.std(returns)
    return mean_reward, std_reward

max_episode_steps = 250
evaluate_ep = 10

# Evaluate BC
env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=max_episode_steps)
reward_mean_bc, reward_std_bc = evaluate_with_stats(bc, env, n_episodes=evaluate_ep)
packet_loss_bc = 1 - reward_mean_bc
print(f"BC avg packet loss rate: {1-reward_mean_bc:.4f} ± {reward_std_bc:.4f}")

# Evaluate IQL
env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=max_episode_steps)
reward_mean_iql, reward_std_iql = evaluate_with_stats(iql, env, n_episodes=evaluate_ep)
packet_loss_iql = 1 - reward_mean_iql
print(f"IQL avg packet loss rate: {1-reward_mean_iql:.4f} ± {reward_std_iql:.4f}")


# Evaluate CQL
env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=max_episode_steps)
reward_mean_cql, reward_std_cql = evaluate_with_stats(cql, env, n_episodes=evaluate_ep)
packet_loss_cql = 1 - reward_mean_cql
print(f"CQL avg packet loss rate: {1-reward_mean_cql:.4f} ± {reward_std_cql:.4f}")


# env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=5000)
# reward_mean_td3bc, reward_std_td3bc = evaluate_with_stats(td3bc, env, n_episodes=evaluate_ep)
# packet_loss_td3bc = 1 - reward_mean_td3bc
# print(f"TD3+BC avg reward: {reward_mean_td3bc:.6f} ± {reward_std_td3bc:.6f}")
# print(f"TD3+BC avg packet loss rate: {packet_loss_td3bc:.6f}")